# 08 — Temporal Anomaly Detection

## Objective
Examine whether temporal velocity features (time since previous transaction per entity) improve anomaly ranking.


In [ ]:

from pathlib import Path
import sys
import pandas as pd
from data_utils import load_processed
from anomaly import fit_isolation_forest
from evaluation import evaluate_anomaly_scores, summary_table

ROOT = Path.cwd()
if not (ROOT / "data").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

X_train = load_processed("X_train")
X_val   = load_processed("X_val")
y_val   = X_val["class"].values

temporal_cols = [c for c in X_train.columns if "time_since" in c or "account_age" in c or "instant" in c or "hour" in c or "weekend" in c or "night" in c]
other_cols = [c for c in X_train.columns if c not in temporal_cols and c != "class"]

results = {}
for label, cols in [("Without temporal", other_cols), ("With temporal", other_cols+temporal_cols)]:
    model = fit_isolation_forest(X_train[cols], contamination=0.09)
    s = model.predict_anomaly_score(X_val[cols])
    results[label] = evaluate_anomaly_scores(y_val, s)
print(summary_table(results))
